# Multimodal RAG Benchmarking with Visual-RAG

Benchmark: **Visual-RAG**

Yin Wu, Quanyu Long, Jing Li, Jianfei Yu, & Wenya Wang. (2025). Visual-RAG: Benchmarking Text-to-Image Retrieval Augmented Generation for Visual Knowledge Intensive Queries.

## Dataset Download

In [3]:
!git clone https://github.com/visual-rag/visual-rag

Cloning into 'visual-rag'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 58 (delta 18), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 3.37 MiB | 5.00 MiB/s, done.
Resolving deltas: 100% (18/18), done.


Image dataset provided by iNaturalist 2021 Competition training dataset

(Due to time and compute constraints, we use the validation dataset from the competition)

In [ ]:
!curl https://ml-inat-competition-datasets.s3.amazonaws.com/2021/val.tar.gz
!curl https://ml-inat-competition-datasets.s3.amazonaws.com/2021/val.json.tar.gz

or if ```aws-cli``` is available, use the following command for faster downloads

In [ ]:
!aws s3 cp s3://ml-inat-competition-datasets/2021/val.tar.gz .
!aws s3 cp s3://ml-inat-competition-datasets/2021/val.json.tar.gz

Unzip compressed files

In [ ]:
!tar -xf val.tar.gz -C .
!tar -xf val.json.tar.gz -C .

## Imports

In [2]:
from config.settings import RAGConfig
from RAGPipeline import RAGSystem
from utils.image_extraction import local_image_to_data_url
from utils.benchmark_eval_helpers import grade_with_llm_judge

import json
import ijson
from mimetypes import guess_type
from tqdm import tqdm
from datetime import datetime
import time
import os
from typing import List, Dict

2026-04-19 18:28:13.683370: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-19 18:28:13.803529: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-19 18:28:21.534911: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/javan/anaconda3/lib/python3.13/site-packages/clip/clip.py:6: UserWarning: pkg_resourc

## Dataset Load

In [3]:
dataset = []

with open('visual-rag/v2_anno.jsonl', 'r') as f:
    for line in f:
        dataset.append(json.loads(line))

dataset[0]

{'images': {'00855_Animalia_Arthropoda_Insecta_Lepidoptera_Cossidae_Prionoxystus_robiniae/83f81be6-213a-4677-9cf4-33542fcbc112.jpg': 0,
  '00855_Animalia_Arthropoda_Insecta_Lepidoptera_Cossidae_Prionoxystus_robiniae/5622cc5c-177c-4a19-816e-c60b3a089653.jpg': 0,
  '00855_Animalia_Arthropoda_Insecta_Lepidoptera_Cossidae_Prionoxystus_robiniae/61703a7a-a43a-4dad-bcdd-f1f483e6a109.jpg': 0,
  '00855_Animalia_Arthropoda_Insecta_Lepidoptera_Cossidae_Prionoxystus_robiniae/ffe73449-cfa4-4f2d-8145-69c8feedece2.jpg': 0,
  '00855_Animalia_Arthropoda_Insecta_Lepidoptera_Cossidae_Prionoxystus_robiniae/4255da3d-f61b-4e42-8d98-7e9195f13570.jpg': 0,
  '00855_Animalia_Arthropoda_Insecta_Lepidoptera_Cossidae_Prionoxystus_robiniae/c20708fc-aa9c-4229-a7a7-728a0208a244.jpg': 0,
  '00855_Animalia_Arthropoda_Insecta_Lepidoptera_Cossidae_Prionoxystus_robiniae/b8d6d8bd-9cf5-4f18-8f0c-4109210e70b0.jpg': 0,
  '00855_Animalia_Arthropoda_Insecta_Lepidoptera_Cossidae_Prionoxystus_robiniae/a1ed479f-f1f3-4f81-941e-e08b

In [4]:
image_dataset = {}

with open('val.json', 'rb') as f:
    for key, value in ijson.kvitems(f, ''):
        image_dataset[key] = value

## Setup

In [ ]:
config = RAGConfig()

rag_system = RAGSystem(config)

## Ingestion

Note that this benchmark uses a pure-visual corpus with annotations.
We hence do not use our API for document ingestion.

In [ ]:
for i in range(len(image_dataset['images'])):
    image_metadata = image_dataset['images'][i]
    
    image_path = image_metadata['file_name']
    image_dir = image_path.split('/')[1]
    
    image_category = next((item for item in image_dataset['categories'] if item.get('image_dir_name') == image_dir), None)

    image_data_url = local_image_to_data_url(image_path=image_path)

    image_caption = f"The species {image_category['name']}, commonly known as {image_category['common_name']}, from the supercategory {image_category['supercategory']}, \
of kingdom {image_category['kingdom']}, phylum {image_category['phylum']}, class {image_category['class']}, order {image_category['order']}, family {image_category['family']}, \
genus {image_category['genus']}, and specific epithet {image_category['specific_epithet']}."

    item_metadata = {
        "page_num": 1,
        "img_index": 1,
        "context": image_caption,
        "image_data_url": image_data_url,
        "caption": image_caption,
        "mime_type": guess_type(image_data_url)[0],
        "source_file": image_path,
        "has_caption": True,
        "stored": False
    }
    image_id = rag_system.image_store.store_image(image_data_url=image_data_url, metadata=item_metadata)
    if image_id:
        item_metadata['image_id'] = image_id

    image_embedding = rag_system.multimodal_embedding.encode_image(image_data_url=image_data_url)
    caption_embedding = rag_system.multimodal_embedding.encode_text(text=image_caption)

    rag_system.multi_vector_store.add_image(image_id=image_id, image_embedding=image_embedding, caption_embedding=caption_embedding, metadata=item_metadata)

## Generating Answers

In [5]:
def generate_benchmark_answers_sync(
        rag_system: RAGSystem,
        dataset: List[Dict],
        output_path: str = "./visual-rag/results.jsonl",
        rate_limit_delay: float = 0.5,
        resume: bool = True
    ) -> List[Dict]:
    """
    Generates answers for all questions in the benchmark dataset.
    
    args:
    - rag_system (RAGSystem): Initialized RAGSystem instance
    - dataset (List[Dict]): a list of benchmark items
    - output_path (str): path to save results
    - rate_limit_delay (float): seconds to wait between LLM calls
    - resume (bool): whether to skip already processed items from output file

    returns:
    - a list of result dictionaries with answers
    """
    processed_questions = set()
    results = []
    
    if resume and os.path.exists(output_path):
        try:
            with open(output_path, 'r') as f:
                for line in f:
                    item = json.loads(line)
                    processed_questions.add(item.get('question', ''))
                    results.append(item)
            print(f"Resumed from {len(results)} previously processed questions\n")
        except Exception as e:
            print(f"Could not resume: {e}. Starting fresh.\n")
    
    total = len(dataset)
    
    with tqdm(total=total, desc="Generating answers", initial=len(results)) as pbar:
        for idx, item in enumerate(dataset):
            # skip if already processed
            if item.get('question') in processed_questions:
                pbar.update(1)
                continue
            
            try:
                question = item.get('question')
                print(f"Question: {question}")
                
                rag_result = rag_system.query(question) 
                
                result = {
                    'qid': idx,
                    'question': question,
                    'answer': rag_result.get('answer', ''),
                    'timestamp': datetime.now().isoformat(),
                    'metadata': {
                        'retrieval_mode': rag_result.get('retrieval_mode'),
                        'documents_retrieved': rag_result.get('retrieval_metadata', {}).get('documents_retrieved', 0),
                        'images_used': rag_result.get('generation_metadata', {}).get('images_used', 0),
                        'image_ids': rag_result.get('retrieved_image_ids')
                    }
                }
                
                if 'answer' in item:
                    result['reference_answer'] = item['answer']
                
                with open(output_path, 'a') as f:
                    json.dump(result, f)
                    f.write('\n')
                
                results.append(result)
                
            except Exception as e:
                print(f"\nError at item {idx}: {str(e)}")
            
            # rate limiting
            time.sleep(rate_limit_delay)
            pbar.update(1)
    
    print(f"\nCompleted! Total processed: {len(results)}")
    
    return results

In [6]:
def run_grading(
        results: List[Dict], 
        llm_client, 
        output_file: str = "./visual-rag/grading_results.json"
    ) -> List[Dict]:
    """
    Grades a list of results using an LLM judge.

    args:
    - results (List[Dict]): a list of dictionaries of generated results
    - llm_client: the LLM client to be used for grading
    - output_file (str): path to save the detailed grading results

    returns:
    - a list of dictionaries containing grading results for each item
    """
    grading_data = []
    for result in results:
        item = {
            'id': result.get('qid', ''),
            'question': result['question'],
            'llm_response': result['answer'],               # generated answer
            'answers': result.get('reference_answer', [])  # ground truth 
        }
        grading_data.append(item)

    grading_results = grade_with_llm_judge(
        responses=grading_data,
        client=llm_client,
        output_file=output_file
    )

    # results summary
    print(f"\n{'='*50}")
    print(f"Grading Summary")
    print(f"{'='*50}")
    print(f"Accuracy: {grading_results['accuracy']:.2%}")
    print(f"Correct: {grading_results['correct_count']}/{grading_results['total_count']}")
    print(f"\nDetailed results saved to: {output_file}")

    return grading_results

GPT-4o-mini, zero-shot

In [7]:
from models.llm_openai_azure import LLM_OpenAI_Azure

llm_client = LLM_OpenAI_Azure(config=config)

In [13]:
def generate_benchmark_answers_zeroshot(
        llm_client,
        dataset: List[Dict],
        output_path: str = "./visual-rag/results.jsonl",
        rate_limit_delay: float = 0.5,
        resume: bool = True
    ) -> List[Dict]:
    """
    Generates answers for all questions in the benchmark dataset.
    
    args:
    - llm_client: the LLM client to be used
    - dataset (List[Dict]): a list of benchmark items
    - output_path (str): path to save results
    - rate_limit_delay (float): seconds to wait between LLM calls
    - resume (bool): whether to skip already processed items from output file

    returns:
    - a list of result dictionaries with answers
    """
    processed_questions = set()
    results = []
    
    if resume and os.path.exists(output_path):
        try:
            with open(output_path, 'r') as f:
                for line in f:
                    item = json.loads(line)
                    processed_questions.add(item.get('question', ''))
                    results.append(item)
            print(f"Resumed from {len(results)} previously processed questions\n")
        except Exception as e:
            print(f"Could not resume: {e}. Starting fresh.\n")
    
    total = len(dataset)
    
    with tqdm(total=total, desc="Generating answers", initial=len(results)) as pbar:
        for idx, item in enumerate(dataset):
            # skip if already processed
            if item.get('question') in processed_questions:
                pbar.update(1)
                continue
            
            try:
                question = item.get('question')
                print(f"Question: {question}")
                
                generated_result = llm_client.generate_response(question)
                
                result = {
                    'qid': idx,
                    'question': question,
                    'answer': generated_result.get('answer', ''),
                    'timestamp': datetime.now().isoformat(),
                    'metadata': {
                        'retrieval_mode': generated_result.get('retrieval_mode'),
                        'documents_retrieved': generated_result.get('retrieval_metadata', {}).get('documents_retrieved', 0),
                        'images_used': generated_result.get('generation_metadata', {}).get('images_used', 0),
                    }
                }
                
                if 'answer' in item:
                    result['reference_answer'] = item['answer']
                
                with open(output_path, 'a') as f:
                    json.dump(result, f)
                    f.write('\n')
                
                results.append(result)
                
            except Exception as e:
                print(f"\nError at item {idx}: {str(e)}")
            
            # rate limiting
            time.sleep(rate_limit_delay)
            pbar.update(1)
    
    print(f"\nCompleted! Total processed: {len(results)}")
    
    return results

In [ ]:
results_zeroshot = generate_benchmark_answers_zeroshot(llm_client=llm_client, dataset=dataset, output_path="./visual-rag/results_zeroshot.jsonl")

In [15]:
grading_results_zeroshot = run_grading(results=results_zeroshot, llm_client=llm_client, output_file="./visual-rag/grading_results_zeroshot.json")


Grading 374 generated responses using LLM judge...


Grading: 100%|██████████| 374/374 [12:33<00:00,  2.01s/it]


Detailed results saved to ./visual-rag/grading_results_zeroshot.json

Grading Summary
Accuracy: 43.05%
Correct: 161/374

Detailed results saved to: ./visual-rag/grading_results_zeroshot.json


GPT-4o-mini, multimodal retrieval, $k$=1

In [ ]:
rag_system.config.top_k = 1

results_k1 = generate_benchmark_answers_sync(rag_system=rag_system, dataset=dataset, output_path="./visual-rag/results_k1.jsonl")

In [9]:
grading_results_k1 = run_grading(results=results_k1, llm_client=llm_client, output_file="./visual-rag/grading_results_k1.json")


Grading 374 generated responses using LLM judge...


Grading: 100%|██████████| 374/374 [12:42<00:00,  2.04s/it]


Detailed results saved to ./visual-rag/grading_results_k1.json

Grading Summary
Accuracy: 7.49%
Correct: 28/374

Detailed results saved to: ./visual-rag/grading_results_k1.json


GPT-4o-mini, multimodal retrieval, $k$=3

In [ ]:
rag_system.config.top_k = 3

results_k3 = generate_benchmark_answers_sync(rag_system=rag_system, dataset=dataset, output_path="./visual-rag/results_k3.jsonl")

In [11]:
grading_results_k3 = run_grading(results=results_k3, llm_client=llm_client, output_file="./visual-rag/grading_results_k3.json")


Grading 374 generated responses using LLM judge...


Grading: 100%|██████████| 374/374 [12:31<00:00,  2.01s/it]


Detailed results saved to ./visual-rag/grading_results_k3.json

Grading Summary
Accuracy: 11.76%
Correct: 44/374

Detailed results saved to: ./visual-rag/grading_results_k3.json


GPT-4o-mini, multimodal retrieval, $k$=5

In [ ]:
rag_system.config.top_k = 5

results_k5 = generate_benchmark_answers_sync(rag_system=rag_system, dataset=dataset, output_path="./visual-rag/results_k5.jsonl")

In [13]:
grading_results_k5 = run_grading(results=results_k5, llm_client=llm_client, output_file="./visual-rag/grading_results_k5.json")


Grading 374 generated responses using LLM judge...


Grading: 100%|██████████| 374/374 [12:27<00:00,  2.00s/it]


Detailed results saved to ./visual-rag/grading_results_k5.json

Grading Summary
Accuracy: 15.24%
Correct: 57/374

Detailed results saved to: ./visual-rag/grading_results_k5.json


GPT-4o-mini, multimodal retrieval, $k$=7

In [ ]:
rag_system.config.top_k = 7

results_k7 = generate_benchmark_answers_sync(rag_system=rag_system, dataset=dataset, output_path="./visual-rag/results_k7.jsonl")

In [15]:
grading_results_k7 = run_grading(results=results_k7, llm_client=llm_client, output_file="./visual-rag/grading_results_k7.json")


Grading 374 generated responses using LLM judge...


Grading: 100%|██████████| 374/374 [12:29<00:00,  2.00s/it]


Detailed results saved to ./visual-rag/grading_results_k7.json

Grading Summary
Accuracy: 14.44%
Correct: 54/374

Detailed results saved to: ./visual-rag/grading_results_k7.json


GPT-4o-mini, multimodal retrieval, $k$=10

In [ ]:
rag_system.config.top_k = 10

results_k10 = generate_benchmark_answers_sync(rag_system=rag_system, dataset=dataset, output_path="./visual-rag/results_k10.jsonl")

In [17]:
grading_results_k10 = run_grading(results=results_k10, llm_client=llm_client, output_file="./visual-rag/grading_results_k10.json")


Grading 374 generated responses using LLM judge...


Grading: 100%|██████████| 374/374 [12:26<00:00,  2.00s/it]


Detailed results saved to ./visual-rag/grading_results_k10.json

Grading Summary
Accuracy: 17.11%
Correct: 64/374

Detailed results saved to: ./visual-rag/grading_results_k10.json


GPT-4o-mini, multimodal retrieval, $k$=15

In [ ]:
rag_system.config.top_k = 15

results_k15 = generate_benchmark_answers_sync(rag_system=rag_system, dataset=dataset, output_path="./visual-rag/results_k15.jsonl")

In [23]:
grading_results_k15 = run_grading(results=results_k15, llm_client=llm_client, output_file="./visual-rag/grading_results_k15.json")


Grading 374 generated responses using LLM judge...


Grading: 100%|██████████| 374/374 [12:50<00:00,  2.06s/it]



Detailed results saved to ./visual-rag/grading_results_k15.json

Grading Summary
Accuracy: 19.25%
Correct: 72/374

Detailed results saved to: ./visual-rag/grading_results_k15.json


GPT-4o-mini, multimodal retrieval, $k$=20

In [ ]:
rag_system.config.top_k = 20

results_k20 = generate_benchmark_answers_sync(rag_system=rag_system, dataset=dataset, output_path="./visual-rag/results_k20.jsonl")

In [26]:
grading_results_k20 = run_grading(results=results_k20, llm_client=llm_client, output_file="./visual-rag/grading_results_k20.json")


Grading 374 generated responses using LLM judge...


Grading: 100%|██████████| 374/374 [12:46<00:00,  2.05s/it]


Detailed results saved to ./visual-rag/grading_results_k20.json

Grading Summary
Accuracy: 17.91%
Correct: 67/374

Detailed results saved to: ./visual-rag/grading_results_k20.json
